In [ ]:
from __future__ import annotations

import os
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm

from notebooks.utils.classification_experiment_utils import (
    DATASETS, SOURCE_SPLITS, availability_row, composition_has_both_classes, composition_id, dataset_subsets,
    append_result_frame, build_feature_parent_cohorts, discover_model_artifacts, ensure_result_exports, load_training_episode, release_frames,
)
from notebooks.utils.feature_aggregation_utils import (
    FeatureSet, ensure_prepared_partition, prepared_partition_path, load_prepared_partition,
)
from notebooks.utils.model_training_utils import (
    fit_feature_vector_model, load_model_artifact, load_model_metadata, save_model_artifact,
    score_feature_vector_model,
)
from notebooks.utils.classification_utils import evaluate_scores

DATA_ROOT = Path(os.environ.get("TEXT_DETECTION_DATA", "D:/Projects/master_diploma/text-detection-data"))
PROCESSED_ROOT = DATA_ROOT / "processed"
RUN_ID = "feature-vector-classification-2026-09-25-logreg-random-forest-max600-min100"
SCORER_LLMS = ("gemma_3_1b_it", "gemma_3_270m_it", "littlelamb", "smollm2_360m")
DATASETS = {"Kaggle": "kaggle_train", "MAGE": "mage_train", "Outfox": "outfox_chatgpt_train", "BBC": "bbc_news_content"}
SOURCE_SPLITS = {"Kaggle": ("HG", "MG"), "MAGE": ("HG", "MG"), "Outfox": ("HG", "MG"), "BBC": ("HG",)}
TRAINING_COMPOSITIONS = dataset_subsets(tuple(DATASETS))
WINDOW_CONFIGURATIONS = ("whole_text", 32, 64, 128)
FEATURE_SETS = {
    "scalar_all": FeatureSet("scalar_all", ("d_score_top20", "perplexity", "prompt_mu_score", "prompt_sigma_score", "d_score_top20_coverage")),
    "scalar_compact": FeatureSet("scalar_compact", ("d_score_top20", "perplexity", "prompt_mu_score")),
    "simple_similarity": FeatureSet("simple_similarity", ("mean_candidate_similarity__mean", "mean_candidate_similarity__std", "top20_coverage__mean", "top20_coverage__std", "observed_logprob__mean", "observed_logprob__std", "mean_candidate_logprob_delta__mean", "mean_candidate_logprob_delta__std")),
    "extended": FeatureSet("extended", ("d_score_top20", "perplexity", "prompt_mu_score", "prompt_sigma_score", "d_score_top20_coverage", "mean_candidate_similarity__mean", "mean_candidate_similarity__std", "top20_coverage__mean", "top20_coverage__std", "observed_logprob__mean", "observed_logprob__std", "mean_candidate_logprob_delta__mean", "mean_candidate_logprob_delta__std")),
}
MODEL_FAMILIES = ("logistic_regression", "random_forest")
TRAINING_PARTITIONS = ("train", "validation")
COHORT_SEED, MAX_PER_SPLIT, MIN_TEXT_TOKENS, STRIDE = 42, 600, 100, 16


In [ ]:
preparation_rows, available = [], {}
for scorer_llm in tqdm(SCORER_LLMS, desc="Preparing feature partitions", unit="scorer"):
    for source_name, dataset in DATASETS.items():
        for split in SOURCE_SPLITS[source_name]:
            for feature_set in FEATURE_SETS.values():
                for window_configuration in tqdm(WINDOW_CONFIGURATIONS, desc=f"{scorer_llm}/{source_name}/{split}/{feature_set.name}", unit="window", leave=False):
                    path = prepared_partition_path(DATA_ROOT, scorer_llm=scorer_llm, dataset=dataset, split=split, feature_set=feature_set, window_configuration=window_configuration)
                    try:
                        partition_exists = path.exists()
                        status = ensure_prepared_partition(DATA_ROOT, PROCESSED_ROOT, scorer_llm=scorer_llm, dataset=dataset, split=split, feature_set=feature_set, window_configuration=window_configuration, stride=STRIDE)
                        preparation_rows.append({**availability_row(stage="preparation", status=status["status"], scorer_llm=scorer_llm, dataset=source_name, split=split, feature_set=feature_set.name, window_configuration=window_configuration, error=status.get("error", "")), "path": str(status["path"])})
                        if status["status"] == "available":
                            available[(scorer_llm, source_name, split, feature_set.name, str(window_configuration))] = Path(status["path"])
                    except (FileNotFoundError, KeyError, ValueError, OSError) as error:
                        preparation_rows.append({**availability_row(stage="preparation", status="unavailable", scorer_llm=scorer_llm, dataset=source_name, split=split, feature_set=feature_set.name, window_configuration=window_configuration, error=error), "path": str(path)})
                        continue
preparation_status = pd.DataFrame(preparation_rows)


In [ ]:
exports, _ = ensure_result_exports(DATA_ROOT, RUN_ID)
reusable_training_keys = set()
for artifact_path in discover_model_artifacts(DATA_ROOT / "experiment-results" / "classification" / "models", filters={"run_id": RUN_ID}):
    try:
        artifact_metadata = load_model_metadata(artifact_path, expected={"run_id": RUN_ID})
        load_model_artifact(artifact_path, expected={"model_family": artifact_metadata["model_family"], "scorer_llm": artifact_metadata["scorer_llm"], "feature_set": artifact_metadata["feature_set"], "feature_columns": artifact_metadata["feature_columns"]}, trust_artifact=True)
        reusable_training_keys.add((str(artifact_metadata.get("training_composition", artifact_metadata.get("training_regime", ""))), str(artifact_metadata["scorer_llm"]), str(artifact_metadata["feature_set"]), str(artifact_metadata["training_window"]), str(artifact_metadata["model_family"])))
    except (FileNotFoundError, KeyError, ValueError, OSError, PermissionError):
        continue
training_metrics, training_coverage, run_summary, artifacts = [], [], [], []
for scorer_llm in tqdm(SCORER_LLMS, desc="Training classifier episodes", unit="scorer"):
    for training_composition in TRAINING_COMPOSITIONS:
        training_composition_id = composition_id(training_composition)
        if not composition_has_both_classes(training_composition, SOURCE_SPLITS):
            run_summary.append({"run_id": RUN_ID, "stage": "training", "status": "unavailable", "scorer_llm": scorer_llm, "dataset": training_composition_id, "training_composition": training_composition_id, "evaluation_composition": "", "split": "", "feature_set": "", "training_window": "", "evaluation_window": "", "model_family": "", "artifact_path": "", "error": "training composition has no HG/MG binary pair"})
            continue
        for feature_set_name, feature_set in FEATURE_SETS.items():
            for window_configuration in WINDOW_CONFIGURATIONS:
                pending_model_families = [model_family for model_family in MODEL_FAMILIES if (training_composition_id, scorer_llm, feature_set_name, str(window_configuration), model_family) not in reusable_training_keys]
                if not pending_model_families:
                    print(f"[classification resume] skipping {training_composition_id}/{scorer_llm}/{feature_set_name}/{window_configuration}: all artifacts are valid")
                    continue
                source_frames, missing_sources = [], []
                for source_name in training_composition:
                    dataset = DATASETS[source_name]
                    paths = [available.get((scorer_llm, source_name, split, feature_set_name, str(window_configuration))) for split in SOURCE_SPLITS[source_name]]
                    if any(path is None for path in paths):
                        missing_sources.append(source_name)
                        continue
                    for split in SOURCE_SPLITS[source_name]:
                        path = available.get((scorer_llm, source_name, split, feature_set_name, str(window_configuration)))
                        try:
                            frame = load_prepared_partition(path, expected={"scorer_llm": scorer_llm, "dataset": dataset, "split": split, "feature_set": feature_set_name, "feature_columns": list(feature_set.source_columns), "window_configuration": "whole_text" if window_configuration == "whole_text" else str(window_configuration), "schema_version": 1, "stride": None if window_configuration == "whole_text" else STRIDE})
                            source_frames.append(frame.assign(dataset=dataset))
                        except (FileNotFoundError, KeyError, ValueError, OSError) as error:
                            missing_sources.append(f"{source_name}: {error}")
                if missing_sources:
                    release_frames(*source_frames)
                    run_summary.append({"run_id": RUN_ID, "stage": "training", "status": "unavailable", "scorer_llm": scorer_llm, "dataset": training_composition_id, "training_composition": training_composition_id, "evaluation_composition": "", "split": "", "feature_set": feature_set_name, "training_window": str(window_configuration), "evaluation_window": "", "model_family": "", "artifact_path": "", "error": f"missing training sources: {missing_sources}"})
                    continue
                parent_frame = pd.concat(source_frames, ignore_index=True)[["dataset", "split", "text_id", "label", "stop_token"]]
                release_frames(*source_frames)
                try:
                    cohorts = build_feature_parent_cohorts(parent_frame, seed=COHORT_SEED, max_per_split=MAX_PER_SPLIT, min_text_tokens=MIN_TEXT_TOKENS)
                    train, validation = load_training_episode(DATA_ROOT, scorer_llm=scorer_llm, datasets=tuple(DATASETS[name] for name in training_composition), feature_set=feature_set_name, window_configuration=window_configuration, feature_columns=feature_set.source_columns, train_cohort=cohorts["train"], validation_cohort=cohorts["validation"])
                except (FileNotFoundError, KeyError, ValueError, OSError) as error:
                    run_summary.append({"run_id": RUN_ID, "stage": "training", "status": "unavailable", "scorer_llm": scorer_llm, "dataset": training_composition_id, "training_composition": training_composition_id, "evaluation_composition": "", "split": "", "feature_set": feature_set_name, "training_window": str(window_configuration), "evaluation_window": "", "model_family": "", "artifact_path": "", "error": str(error)})
                    release_frames(parent_frame)
                    continue
                for model_family in tqdm(pending_model_families, desc=f"{training_composition_id}/{scorer_llm}/{feature_set_name}/{window_configuration} models", unit="model", leave=False):
                    try:
                        fitted = fit_feature_vector_model(model_family, train, feature_columns=feature_set.source_columns, seed=COHORT_SEED)
                        scores = score_feature_vector_model(fitted, validation)
                        metrics = evaluate_scores(scores["label"], scores["mg_score"], 0.5)
                    except (FileNotFoundError, KeyError, ValueError, OSError) as error:
                        run_summary.append({"run_id": RUN_ID, "stage": "training", "status": "unavailable", "scorer_llm": scorer_llm, "dataset": training_composition_id, "training_composition": training_composition_id, "evaluation_composition": "", "split": "", "feature_set": feature_set_name, "training_window": str(window_configuration), "evaluation_window": "", "model_family": model_family, "artifact_path": "", "error": str(error)})
                        continue
                    training_metrics.append({"model_family": model_family, "scorer_llm": scorer_llm, "feature_set": feature_set_name, "training_regime": training_composition_id, "training_composition": training_composition_id, "training_window": str(window_configuration), "partition": "validation", "sample_count": len(validation), **{key: metrics[key] for key in ("accuracy", "balanced_accuracy", "macro_f1", "roc_auc")}})
                    training_coverage.append({"dataset": training_composition_id, "training_composition": training_composition_id, "split": "HG+MG", "class": "binary", "parent_count": len(cohorts["train"]) + len(cohorts["validation"]), "row_count": len(train) + len(validation), "availability_status": "available"})
                    metadata = {"artifact_version": 1, "run_id": RUN_ID, "created_at": datetime.now(timezone.utc).isoformat(), "model_family": model_family, "scorer_llm": scorer_llm, "training_datasets": list(DATASETS[name] for name in training_composition), "training_regime": training_composition_id, "training_composition": training_composition_id, "training_window": str(window_configuration), "feature_set": feature_set_name, "feature_columns": list(feature_set.source_columns), "cohort_seed": COHORT_SEED, "max_per_split": MAX_PER_SPLIT, "min_text_tokens": MIN_TEXT_TOKENS, "stride": None if window_configuration == "whole_text" else STRIDE, "train_row_count": len(train), "validation_row_count": len(validation), "train_parent_count": len(cohorts["train"]), "validation_parent_count": len(cohorts["validation"]), "train_hg_row_count": int(train["label"].eq(0).sum()), "train_mg_row_count": int(train["label"].eq(1).sum()), "validation_hg_row_count": int(validation["label"].eq(0).sum()), "validation_mg_row_count": int(validation["label"].eq(1).sum())}
                    try:
                        artifact_path = save_model_artifact(DATA_ROOT, fitted, metadata=metadata, run_id=RUN_ID, training_metrics=pd.DataFrame(training_metrics[-1:]), training_coverage=pd.DataFrame(training_coverage[-1:]))
                    except (FileNotFoundError, KeyError, ValueError, OSError) as error:
                        run_summary.append({"run_id": RUN_ID, "stage": "training", "status": "unavailable", "scorer_llm": scorer_llm, "dataset": training_composition_id, "training_composition": training_composition_id, "evaluation_composition": "", "split": "", "feature_set": feature_set_name, "training_window": str(window_configuration), "evaluation_window": "", "model_family": model_family, "artifact_path": "", "error": str(error)})
                        continue
                    artifacts.append(artifact_path)
                    run_summary.append({"run_id": RUN_ID, "stage": "training", "status": "available", "scorer_llm": scorer_llm, "dataset": training_composition_id, "training_composition": training_composition_id, "evaluation_composition": "", "split": "HG+MG", "feature_set": feature_set_name, "training_window": str(window_configuration), "evaluation_window": "", "model_family": model_family, "artifact_path": str(artifact_path), "error": ""})
                release_frames(parent_frame, train, validation, scores)


In [ ]:
result_root = DATA_ROOT / "experiment-results" / "classification" / "runs"
result_root.mkdir(parents=True, exist_ok=True)
run_summary = pd.DataFrame(run_summary)
result_frames = {
    "training_metrics": pd.DataFrame(training_metrics),
    "training_coverage": pd.DataFrame(training_coverage),
    "evaluation_metrics": pd.DataFrame(),
    "evaluation_predictions": pd.DataFrame(),
    "evaluation_coverage": pd.DataFrame(),
    "run_summary": run_summary,
}
for name, frame in result_frames.items():
    print(f"[classification export] {name}: rows={len(frame)}, columns={list(frame.columns)}")
if not run_summary.empty:
    print(f"[classification export] run statuses: {run_summary['status'].value_counts(dropna=False).to_dict()}")
    errors = run_summary.loc[run_summary["error"].astype(bool), ["stage", "scorer_llm", "feature_set", "training_window", "model_family", "error"]]
    if not errors.empty:
        print("[classification export] recorded errors:")
        display(errors)
for name, frame in result_frames.items():
    append_result_frame(exports[name], frame) if name in exports else None
append_result_frame(result_root / f"{RUN_ID}-preparation.csv", preparation_status)
append_result_frame(result_root / f"{RUN_ID}-summary.csv", run_summary)
exports
